# Convert crop production data to point-in-time

Convert SPAM annual crop production data into annual "point in time" estimates. This is done by dividing the total annual production by the average number of harvests for a particular geographic area, crop type and system (e.g. irrigated).

### Input data

In [8]:
# All crop data
SPAM_BASE_DIR = "../../data/raw/SPAM_2010"

# Reference crop tables
CROP_INTENSITY_CSV = "../../data/external/cropping_intensity_all_augmented.csv"
SPAM_CROP_LIST_CSV = "../../data/external/spam_crop_list.csv"


### Method

**(A) Prepare crop intensity reference table**
1. Load in a pre-generated table of crop intensities (harvest frequencies over a year), organised by area, crop type and system (e.g. irrigated)

**(B) Update production based on crop intensities**
1. Iterate through SPAM_BASE_DIR CSV files (contains one CSV for each production system). Note that we use the CSV data because it contains columns (name_admin, name_cntr) which easily allow us to cross reference with the equivalent columns in the crop intensity table. This does however raise complications later with translating this data into raster format
2. Calculate new production values for each pixel based on (A)

### (A) Prepare crop intensity reference table

In [9]:
import pandas as pd

# Process crop intensity
crop_intensity = pd.read_csv(CROP_INTENSITY_CSV)

# Create one dataframe for each crop production system, this allows us to create 1-1
# relationships between a crop intensity lookup table and a production data file 
# (which are saved according to production system)

crop_intensity["name_admin"] = crop_intensity["name_admin"].apply(str.strip)
crop_intensity["name_cntr"] = crop_intensity["name_cntr"].apply(str.strip)

crop_intensity["name_admin_lower"] = crop_intensity.name_admin.apply(str.lower)
crop_intensity["name_cntr_lower"] = crop_intensity.name_cntr.apply(str.lower)

crop_intensity_irrigated = crop_intensity[crop_intensity.rec_type == 'CIIRR']
crop_intensity_rainfed_high_inputs = crop_intensity[crop_intensity.rec_type == 'CIRFH']
crop_intensity_rainfed_low_inputs = crop_intensity[crop_intensity.rec_type == 'CIRFL']
crop_intensity_rainfed_subsistence = crop_intensity[crop_intensity.rec_type == 'CIRFL']

In [10]:
crop_intensity_irrigated

,iso3,prod_level,name_cntr,name_admin,rec_type,unit,wheat,rice,maize,barley,...,banana,plantain,trop_fruit,temp_fruit,vegetable,rest_crop,year_data,source,name_admin_lower,name_cntr_lower
0,AFG,AF00,AFGHANISTAN,AFGHANISTAN,CIIRR,nr,1.27,1.27,1.27,1.0,...,1.0,1.0,1.00,1.00,1.00,1.00,NaN,NaN,afghanistan,afghanistan
3,BGD,BG00,Bangladesh,BANGLADESH,CIIRR,nr,1.41,1.60,1.00,1.0,...,1.0,0.0,1.23,1.49,1.43,1.26,NaN,NaN,bangladesh,bangladesh
7,BGD,BG01,BANGLADESH,Barisal,CIIRR,nr,1.00,2.22,1.00,1.0,...,1.0,1.0,2.00,2.00,2.00,2.00,NaN,NaN,barisal,bangladesh
11,BGD,BG02,BANGLADESH,Chittagong,CIIRR,nr,1.00,1.27,1.00,1.0,...,1.0,1.0,1.00,1.00,1.00,1.00,NaN,NaN,chittagong,bangladesh
15,BGD,BG03,BANGLADESH,Dhaka,CIIRR,nr,1.00,1.33,1.00,1.0,...,1.0,1.0,1.00,1.00,1.50,1.00,NaN,NaN,dhaka,bangladesh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4180,USA,US47,united states of america,Virginia,CIIRR,nr,1.00,1.00,1.00,1.0,...,1.0,1.0,1.00,1.00,1.06,1.00,"avg(02,07)",USDA,virginia,united states of america
4183,USA,US48,united states of america,Washington,CIIRR,nr,1.55,1.00,1.00,1.0,...,1.0,1.0,1.00,1.00,1.03,1.00,"avg(02,07)",USDA,washington,united states of america
4186,USA,US49,united states of america,West Virginia,CIIRR,nr,1.00,1.00,1.00,1.0,...,1.0,1.0,1.00,1.00,1.01,1.00,"avg(02,07)",USDA,west virginia,united states of america
4189,USA,US50,united states of america,Wisconsin,CIIRR,nr,1.00,1.00,1.00,1.0,...,1.0,1.0,1.00,1.00,1.02,1.00,"avg(02,07)",USDA,wisconsin,united states of america


### (B) Update production based on crop intensities

Calculate residues
want these files:

- *_TI	irrigated portion of crop
- *_TH	rainfed high inputs portion of crop
- *_TL	rainfed low inputs portion of crop
- *_TS	rainfed subsistence portion of crop


From Ulrike:

"The column rec_type identifies the production system for which the next columns are valid:

CIIRR: cropping intensity for irrigated crops

CIRFH: cropping intensity for rainfed high-input crops

CIRFL: cropping intensity for rainfed low-input crops

For subsistence production systems we take CIRFL."

In [ ]:
import pandas as pd
from functools import partial
from tqdm import tqdm

def _get_crop_intensity_from_key(key, crop_long, intensity_table):
    """
    Find the "crop intensity" value for a given country + region + crop combination. 
    Country and region are given by `key`, crop is given by `crop_long`, and lookup
    table is a table of values for a single crop production system (e.g. irrigated)
    which gives us crop intensities.
    
    Assumes that for entries where admin name == country name, the first entry in crop
    intensity table is the default value for the whole country. e.g. Mexico, Mexico
    appears twice, and I'm assuming the first entry is the Mexico-country-default, and
    the second entry is for the state of Mexico in the country Mexico.
    """
    name_cntr_lower, name_admin_lower = key.split("#")

    result = crop_intensity_table[crop_long].get(
        (name_cntr_lower, name_admin_lower),  # index to search for
        pd.Series(dtype='object')  # default value if index can't be found
    )

    if not result.empty:
        return result[-1]  # last entry with these values
    else:
        try:
            return intensity_table[crop_long].get(
                (name_cntr_lower, name_cntr_lower)
            )[0]  # default to default entry for country
        except:
            print(name_cntr_lower)
            raise ValueError

            
def _add_intensity_column(grp, crop_long, intensity_table):
    """
    Creates a crop intensity column on a group
    """
    grp[f'{crop_long}_intensity'] = _get_crop_intensity_from_key(grp.name, crop_long, intensity_table)
    return grp    
    

# Crop names
spam_croplist = pd.read_csv(SPAM_CROP_LIST_CSV)
spam_crop_mappings = list(zip(list(spam_croplist.spam_short), list(spam_croplist.spam_long)))


# Update production based on crop intensity
prod_dfs_to_save = []

for prod_table, crop_intensity_table in [
    ("spam2010V2r0_global_P_TI.csv", crop_intensity_irrigated),
    ("spam2010V2r0_global_P_TH.csv", crop_intensity_rainfed_high_inputs),
    ("spam2010V2r0_global_P_TL.csv", crop_intensity_rainfed_low_inputs),
    ("spam2010V2r0_global_P_TS.csv", crop_intensity_rainfed_subsistence)
]:
    
    # Get prod system code from file
    # i = irrigated, h = rainfed high inputs, l = rainfed low inputs, s = subsistence
    prod_sys_code = prod_table.split(".")[0][-1].lower()
    
    # Index our crop intensity table to make lookups quick (done in the loop below)
    crop_intensity_table = crop_intensity_table.set_index(["name_cntr_lower", "name_admin_lower"])
    crop_intensity_table.sort_index()
    
    # Load production table (~800k rows)
    prod_df = pd.read_csv(f"{SPAM_BASE_DIR}/spam2010v2r0_global_prod.csv/{prod_table}")

    # Create join attributes
    prod_df["name_admin"] = prod_df["name_adm1"].apply(str.strip)
    prod_df["name_cntr"] = prod_df["name_cntr"].apply(str.strip)
    prod_df["name_admin_lower"] = prod_df.name_admin.apply(str.lower)
    prod_df["name_cntr_lower"] = prod_df.name_cntr.apply(str.lower)
    prod_df["crop_intensity_key"] = prod_df["name_cntr_lower"] + "#" + prod_df["name_admin_lower"]
    
    for crop_short, crop_long in tqdm(spam_crop_mappings):
        
        # In order to make this go reasonably fast: we first calculate crop intensities for each group
        # of like countries + regions (given by crop_intensity_key), and broadcast results to all members
        # of that group.
        func = partial(_add_intensity_column, crop_long=crop_long, intensity_table=crop_intensity_table)
        prod_df = prod_df.groupby('crop_intensity_key').apply(func)
        
        # We then produce a point-in-time (pit) production estimate for each row, by dividing by the
        # crop intensity value for that row
        prod_df[f"{crop_long}_pit"] = prod_df[f"{crop_short}_{prod_sys_code}"] / prod_df[f'{crop_long}_intensity']
        
    prof_df.to_csv(f"../data/processed/production_pit_{prod_sys_code}.csv")

In [ ]:
# We'll try first rasterizing 

# image = features.rasterize(
#             ((g, 255) for g, v in shapes),
#             out_shape=src.shape,
#             transform=src.transform)

# Globally summarise point in time production data

Take the production outputs from above and add them together to produce a global point-in-time production estimate for each crop for 2010. Then, summarise the data.

### Input data

In [1]:
# Production tables
IRRIGATED_PROD_CSV = "../../data/processed/production_pit_i.csv"
RAINFED_HIGH_INPUTS_PROD_CSV = "../../data/processed/production_pit_h.csv"
RAINFED_LOW_INPUTS_PROD_CSV = "../../data/processed/production_pit_l.csv"
SUBSISTENCE_PROD_CSV = "../../data/processed/production_pit_s.csv"

# Crops
SPAM_CROP_LIST_CSV = "../../data/external/spam_crop_list.csv"

In [3]:
import pandas as pd
from tqdm import tqdm

summ_i_df = pd.DataFrame()
i_df = pd.read_csv(IRRIGATED_PROD_CSV)
h_df = pd.read_csv(RAINFED_HIGH_INPUTS_PROD_CSV)
l_df = pd.read_csv(RAINFED_LOW_INPUTS_PROD_CSV)
s_df = pd.read_csv(SUBSISTENCE_PROD_CSV)

In [ ]:
total = pd.concat([
    i_df, h_df, l_df, s_df
])

In [ ]:
total.to_csv("../../data/processed/total_pit.csv")

### Summarise

In [4]:
import pandas as pd
from tqdm import tqdm

summ_i_df = pd.DataFrame()
i_df = pd.read_csv(IRRIGATED_PROD_CSV)
h_df = pd.read_csv(RAINFED_HIGH_INPUTS_PROD_CSV)
l_df = pd.read_csv(RAINFED_LOW_INPUTS_PROD_CSV)
s_df = pd.read_csv(SUBSISTENCE_PROD_CSV)

spam_croplist = pd.read_csv(SPAM_CROP_LIST_CSV)
spam_crop_mappings = list(zip(list(spam_croplist.spam_short), list(spam_croplist.spam_long)))

for short, long in tqdm(spam_crop_mappings):
    
    sum_i = i_df[f"{long}_pit"].sum()
    sum_h = h_df[f"{long}_pit"].sum()
    sum_l = l_df[f"{long}_pit"].sum()
    sum_s = s_df[f"{long}_pit"].sum()
    
    summ_i_df[f"{long}_total"] = [sum_i + sum_h + sum_l + sum_s]

100%|██████████| 42/42 [00:01<00:00, 31.28it/s]


In [5]:
summ_i_df.head()

,wheat_total,rice_total,maize_total,barley_total,pearlmill_total,smallmill_total,sorghum_total,oth_cereal_total,potato_total,sweet_pot_total,...,rob_coffee_total,cocoa_total,tea_total,tobacco_total,banana_total,plantain_total,trop_fruit_total,temp_fruit_total,vegetable_total,rest_crop_total
0,5.925107e+08,4.724108e+08,7.995846e+08,1.346917e+08,2.154706e+07,4.744467e+06,5.565514e+07,6.077396e+07,3.433850e+08,1.015660e+08,...,3745228.6,4384396.6,4553140.7,7.159087e+06,1.049260e+08,2.772592e+07,3.625439e+08,2.464116e+08,8.963313e+08,3.566771e+07


In [6]:
i_df.shape

(832827, 146)

In [7]:
h_df.shape

(832827, 146)

In [ ]:
summ_i_df.transpose().to_csv("../data/processed/production_pit_summary.csv")

In [ ]:
summ_i_df.transpose()